# Comprehensive VLM Evaluation Analysis
## SmolVLM2-500M vs LFM2-VL-450M vs InternVL3.5-1B

This notebook analyzes evaluation results across 16 benchmarks to identify where each model excels and where it lags behind.

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import matplotlib.pyplot as plt
import matplotlib
from matplotlib.patches import FancyBboxPatch
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'sans-serif',
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
})

COLORS = {
    'SmolVLM2-500M': '#2196F3',
    'LFM2-VL-450M': '#FF9800',
    'InternVL3.5-1B': '#4CAF50',
}

MODEL_SHORT = {
    'SmolVLM2-500M': 'SmolVLM2',
    'LFM2-VL-450M': 'LFM2',
    'InternVL3.5-1B': 'InternVL3.5',
}

ROOT = '/home/rishabh/VLMEvalKit'
PATHS = {
    'SmolVLM2-500M': os.path.join(ROOT, 'outputs/SmolVLM2-500M/T20260409_G637ce464'),
    'LFM2-VL-450M': os.path.join(ROOT, 'outputs/LFM2-VL-450M'),
    'InternVL3.5-1B': os.path.join(ROOT, 'outputs/InternVL3_5-1B'),
}

FILE_PREFIX = {
    'SmolVLM2-500M': 'SmolVLM2-500M',
    'LFM2-VL-450M': 'LFM2-VL-450M',
    'InternVL3.5-1B': 'InternVL3_5-1B',
}

MODELS = list(PATHS.keys())
print('Models:', MODELS)
print('Paths verified:', all(os.path.isdir(p) for p in PATHS.values()))

## 1. Load All Aggregate Scores

In [ ]:
def read_overall(path, prefix, benchmark, score_type='acc'):
    """Read Overall score from a _acc.csv or _score.csv file."""
    if score_type == 'json':
        fp = os.path.join(path, f'{prefix}_{benchmark}_score.json')
        if not os.path.exists(fp):
            return None
        with open(fp) as f:
            return json.load(f)
    
    suffixes = {
        'acc': f'{prefix}_{benchmark}_acc.csv',
        'score': f'{prefix}_{benchmark}_score.csv',
        'mathvista': f'{prefix}_{benchmark}_chatgpt-0125_score.csv',
    }
    fp = os.path.join(path, suffixes[score_type])
    if not os.path.exists(fp):
        return None
    df = pd.read_csv(fp)
    return df

def get_overall_score(df_or_dict, benchmark=None):
    """Extract the single 'Overall' number, normalized to 0-100."""
    if df_or_dict is None:
        return np.nan
    if isinstance(df_or_dict, dict):
        v = df_or_dict.get('Final Score Norm', df_or_dict.get('Final Score'))
        return float(v)
    df = df_or_dict
    if 'Overall' in df.columns:
        val = df['Overall'].iloc[0]
        if benchmark == 'MME':
            return float(val)  # MME uses raw totals
        return float(val) * 100 if float(val) <= 1.0 else float(val)
    if 'acc' in df.columns:
        val = df['acc'].iloc[0]
        return float(val) * 100 if float(val) <= 1.0 else float(val)
    return np.nan


BENCHMARKS_META = [
    ('AI2D_TEST',         'acc',       'Knowledge'),
    ('BLINK',             'acc',       'Perception'),
    ('ChartQA_TEST',      'acc',       'Document'),
    ('DocVQA_VAL',        'acc',       'Document'),
    ('InfoVQA_VAL',       'acc',       'Document'),
    ('MMBench_DEV_EN',    'acc',       'General'),
    ('MME',               'score',     'General'),
    ('MMMU_DEV_VAL',      'acc',       'Knowledge'),
    ('MMStar',            'acc',       'General'),
    ('MathVista_MINI',    'mathvista', 'Reasoning'),
    ('OCRBench',          'json',      'OCR'),
    ('POPE',              'score',     'Hallucination'),
    ('RealWorldQA',       'acc',       'Perception'),
    ('SEEDBench_IMG',     'acc',       'General'),
    ('ScienceQA_TEST',    'acc',       'Knowledge'),
    ('TextVQA_VAL',       'acc',       'OCR'),
]

rows = []
for bench, stype, category in BENCHMARKS_META:
    row = {'Benchmark': bench, 'Category': category}
    for model in MODELS:
        raw = read_overall(PATHS[model], FILE_PREFIX[model], bench, stype)
        if bench == 'MME' and raw is not None:
            row[model] = float(raw['perception'].iloc[0]) + float(raw['reasoning'].iloc[0])
        else:
            row[model] = get_overall_score(raw, bench)
    rows.append(row)

overall_df = pd.DataFrame(rows)

display_df = overall_df.copy()
display_df['Best Model'] = display_df[MODELS].idxmax(axis=1)
display_df['Gap (Best - Worst)'] = display_df[MODELS].max(axis=1) - display_df[MODELS].min(axis=1)
display_df = display_df.sort_values('Gap (Best - Worst)', ascending=False)

print(f"{'Benchmark':<20} {'SmolVLM2':>10} {'LFM2':>10} {'InternVL3.5':>12}  {'Best':>14}  {'Gap':>6}")
print('-' * 80)
for _, r in display_df.iterrows():
    vals = [r[m] for m in MODELS]
    best = r['Best Model']
    gap = r['Gap (Best - Worst)']
    fmt = lambda v: f'{v:>10.2f}' if not np.isnan(v) else f'{"N/A":>10}'
    print(f"{r['Benchmark']:<20} {fmt(vals[0])} {fmt(vals[1])} {fmt(vals[2]):>12}  {MODEL_SHORT[best]:>14}  {gap:>6.1f}")

overall_df

## 2. Overall Benchmark Comparison

In [ ]:
# Bar chart of all benchmarks (excluding MME which has a different scale)
plot_df = overall_df[overall_df['Benchmark'] != 'MME'].copy()
plot_df = plot_df.sort_values('InternVL3.5-1B', ascending=True)

fig, ax = plt.subplots(figsize=(12, 8))
y = np.arange(len(plot_df))
h = 0.25

for i, model in enumerate(MODELS):
    vals = plot_df[model].values
    bars = ax.barh(y + i * h, vals, h, label=MODEL_SHORT[model], color=COLORS[model], alpha=0.85)
    for bar, val in zip(bars, vals):
        if not np.isnan(val):
            ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
                    f'{val:.1f}', va='center', fontsize=7, color=COLORS[model])

ax.set_yticks(y + h)
ax.set_yticklabels(plot_df['Benchmark'])
ax.set_xlabel('Score')
ax.set_title('Benchmark Scores Comparison (Higher is Better)')
ax.legend(loc='lower right', framealpha=0.9)
ax.set_xlim(0, 105)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Radar chart for normalized scores (excluding MME)
radar_df = overall_df[overall_df['Benchmark'] != 'MME'].dropna(subset=MODELS).copy()
benchmarks = radar_df['Benchmark'].tolist()
N = len(benchmarks)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(9, 9), subplot_kw=dict(polar=True))

for model in MODELS:
    values = radar_df[model].tolist()
    values += values[:1]
    ax.plot(angles, values, 'o-', linewidth=2, label=MODEL_SHORT[model], color=COLORS[model], markersize=4)
    ax.fill(angles, values, alpha=0.08, color=COLORS[model])

ax.set_xticks(angles[:-1])
labels = [b.replace('_', '\n') for b in benchmarks]
ax.set_xticklabels(labels, fontsize=8)
ax.set_ylim(0, 100)
ax.set_title('Radar: Model Performance Across Benchmarks', pad=20, fontsize=14)
ax.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1), framealpha=0.9)
plt.tight_layout()
plt.show()

In [ ]:
# MME comparison (separate scale - perception + reasoning breakdown)
mme_data = {}
for model in MODELS:
    raw = read_overall(PATHS[model], FILE_PREFIX[model], 'MME', 'score')
    mme_data[model] = raw.iloc[0].to_dict()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for idx, (metric, title) in enumerate([('perception', 'MME Perception'), ('reasoning', 'MME Reasoning')]):
    ax = axes[idx]
    vals = [mme_data[m][metric] for m in MODELS]
    bars = ax.bar([MODEL_SHORT[m] for m in MODELS], vals, color=[COLORS[m] for m in MODELS], alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
                f'{val:.0f}', ha='center', fontsize=10)
    ax.set_title(title)
    ax.set_ylabel('Score')
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('MME Benchmark Breakdown', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

# MME sub-task breakdown
subtasks = [c for c in mme_data[MODELS[0]].keys() if c not in ('perception', 'reasoning')]
fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(len(subtasks))
w = 0.25
for i, model in enumerate(MODELS):
    vals = [mme_data[model][s] for s in subtasks]
    ax.bar(x + i * w, vals, w, label=MODEL_SHORT[model], color=COLORS[model], alpha=0.85)

ax.set_xticks(x + w)
ax.set_xticklabels([s.replace('_', '\n') for s in subtasks], fontsize=8, rotation=45, ha='right')
ax.set_ylabel('Score')
ax.set_title('MME Sub-task Breakdown')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Category-Level Analysis
Group benchmarks by capability area to see high-level strengths.

In [ ]:
cat_df = overall_df[overall_df['Benchmark'] != 'MME'].copy()
cat_scores = cat_df.groupby('Category')[MODELS].mean()

fig, ax = plt.subplots(figsize=(10, 5))
cat_scores.plot(kind='bar', ax=ax, color=[COLORS[m] for m in MODELS], alpha=0.85, width=0.7)
ax.set_ylabel('Average Score')
ax.set_title('Average Score by Capability Category')
ax.set_xticklabels(cat_scores.index, rotation=30, ha='right')
ax.legend([MODEL_SHORT[m] for m in MODELS])
ax.grid(axis='y', alpha=0.3)

for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=7, padding=2)

plt.tight_layout()
plt.show()

print('\nCategory averages:')
cat_scores.columns = [MODEL_SHORT[m] for m in MODELS]
display(cat_scores.round(2))

## 4. Deep-Dive: Subcategory Heatmaps
For benchmarks with rich subcategory breakdowns, let's compare models at a granular level.

In [ ]:
def load_subcategory_csv(benchmark, score_type='acc'):
    """Load the full subcategory CSV for all 3 models, return a tidy DataFrame."""
    suffixes = {
        'acc': f'_{benchmark}_acc.csv',
        'score': f'_{benchmark}_score.csv',
        'mathvista': f'_{benchmark}_chatgpt-0125_score.csv',
    }
    frames = []
    for model in MODELS:
        fp = os.path.join(PATHS[model], FILE_PREFIX[model] + suffixes[score_type])
        if not os.path.exists(fp):
            continue
        df = pd.read_csv(fp)
        df['model'] = model
        frames.append(df)
    if not frames:
        return None
    return pd.concat(frames, ignore_index=True)


def plot_subcategory_heatmap(benchmark, score_type='acc', exclude_cols=None, title=None, normalize=True):
    """Create a heatmap of subcategory scores across models."""
    df = load_subcategory_csv(benchmark, score_type)
    if df is None:
        print(f'No data for {benchmark}')
        return
    
    skip = {'split', 'model', 'Overall', 'tot', 'prefetch', 'hit', 'prefetch_rate'}
    if exclude_cols:
        skip.update(exclude_cols)
    
    if score_type == 'mathvista':
        subcols = [c for c in df.columns if c not in skip and c != 'Task&Skill' and c != 'acc']
        if not subcols:
            subcols = ['acc']
            df_pivot = df.set_index(['model', 'Task&Skill'])['acc'].unstack('Task&Skill')
            df_pivot = df_pivot.drop(columns=['Overall'], errors='ignore')
        else:
            return
    else:
        subcols = [c for c in df.columns if c not in skip]
        if 'split' in df.columns:
            if 'validation' in df['split'].values:
                df = df[df['split'] == 'validation']
            elif 'Overall' in df['split'].values:
                df = df[df['split'] == 'Overall']
            else:
                df = df.iloc[df.groupby('model').apply(lambda x: x.index[0])]
        
        df_pivot = df.set_index('model')[subcols].astype(float)
        if normalize and (df_pivot.max().max() <= 1.0):
            df_pivot = df_pivot * 100
    
    df_pivot.index = [MODEL_SHORT.get(m, m) for m in df_pivot.index]
    
    fig, ax = plt.subplots(figsize=(max(12, len(df_pivot.columns) * 0.8), 3.5))
    im = ax.imshow(df_pivot.values, cmap='RdYlGn', aspect='auto', vmin=df_pivot.values.min() * 0.9)
    
    ax.set_xticks(range(len(df_pivot.columns)))
    col_labels = [c.replace('_', '\n') for c in df_pivot.columns]
    ax.set_xticklabels(col_labels, rotation=60, ha='right', fontsize=8)
    ax.set_yticks(range(len(df_pivot.index)))
    ax.set_yticklabels(df_pivot.index)
    
    for i in range(len(df_pivot.index)):
        for j in range(len(df_pivot.columns)):
            val = df_pivot.values[i, j]
            ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=7,
                    color='white' if val < df_pivot.values.mean() * 0.7 else 'black')
    
    plt.colorbar(im, ax=ax, shrink=0.8)
    ax.set_title(title or f'{benchmark} — Subcategory Comparison')
    plt.tight_layout()
    plt.show()
    
    return df_pivot

In [ ]:
# MMMU subcategory heatmap (coarse groups)
mmmu_coarse_cols = ['Art & Design', 'Business', 'Health & Medicine', 'Humanities & Social Science', 'Science', 'Tech & Engineering']

df = load_subcategory_csv('MMMU_DEV_VAL', 'acc')
df = df[df['split'] == 'validation']
df_pivot = df.set_index('model')[mmmu_coarse_cols].astype(float) * 100
df_pivot.index = [MODEL_SHORT.get(m, m) for m in df_pivot.index]

fig, ax = plt.subplots(figsize=(12, 3.5))
im = ax.imshow(df_pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(df_pivot.columns)))
ax.set_xticklabels([c.replace(' & ', '\n& ') for c in df_pivot.columns], fontsize=9)
ax.set_yticks(range(len(df_pivot.index)))
ax.set_yticklabels(df_pivot.index)
for i in range(len(df_pivot.index)):
    for j in range(len(df_pivot.columns)):
        val = df_pivot.values[i, j]
        ax.text(j, i, f'{val:.1f}', ha='center', va='center', fontsize=10,
                color='white' if val < 35 else 'black')
plt.colorbar(im, ax=ax, shrink=0.8)
ax.set_title('MMMU (validation) — Coarse Category Breakdown')
plt.tight_layout()
plt.show()

In [ ]:
_ = plot_subcategory_heatmap('MMBench_DEV_EN', 'acc', title='MMBench — Capability Breakdown')

In [ ]:
_ = plot_subcategory_heatmap('AI2D_TEST', 'acc', title='AI2D — Topic Breakdown')

In [ ]:
_ = plot_subcategory_heatmap('SEEDBench_IMG', 'acc', title='SEEDBench — Category Breakdown')

In [ ]:
_ = plot_subcategory_heatmap('BLINK', 'acc', title='BLINK — Visual Reasoning Subcategory Breakdown')

In [ ]:
# OCRBench subcategory comparison
ocr_data = {}
for model in MODELS:
    fp = os.path.join(PATHS[model], FILE_PREFIX[model] + '_OCRBench_score.json')
    with open(fp) as f:
        ocr_data[model] = json.load(f)

ocr_cats = [k for k in ocr_data[MODELS[0]].keys() if k not in ('Final Score', 'Final Score Norm')]
ocr_df = pd.DataFrame({MODEL_SHORT[m]: [ocr_data[m][c] for c in ocr_cats] for m in MODELS}, index=ocr_cats)

fig, ax = plt.subplots(figsize=(12, 4))
ocr_df.plot(kind='bar', ax=ax, color=[COLORS[m] for m in MODELS], alpha=0.85, width=0.7)
ax.set_title('OCRBench — Subcategory Scores')
ax.set_ylabel('Score')
ax.set_xticklabels([c.replace(' ', '\n') for c in ocr_cats], rotation=30, ha='right', fontsize=9)
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

In [ ]:
# MathVista skill breakdown (InternVL and LFM have scores)
mathvista_frames = []
for model in MODELS:
    fp = os.path.join(PATHS[model], FILE_PREFIX[model] + '_MathVista_MINI_chatgpt-0125_score.csv')
    if os.path.exists(fp):
        df = pd.read_csv(fp)
        df['model'] = MODEL_SHORT[model]
        mathvista_frames.append(df)

if len(mathvista_frames) >= 2:
    mv_df = pd.concat(mathvista_frames)
    skills = mv_df[mv_df['Task&Skill'] != 'Overall']
    pivot = skills.pivot(index='Task&Skill', columns='model', values='acc')
    
    fig, ax = plt.subplots(figsize=(12, 5))
    pivot.plot(kind='barh', ax=ax, color=[COLORS[m] for m in MODELS if MODEL_SHORT[m] in pivot.columns], alpha=0.85)
    ax.set_xlabel('Accuracy (%)')
    ax.set_title('MathVista — Skill Breakdown')
    ax.grid(axis='x', alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print('MathVista scored results only available for', len(mathvista_frames), 'model(s)')

## 5. Pairwise Gap Analysis
Identify where each model falls behind the best performer and by how much.

In [ ]:
gap_df = overall_df[overall_df['Benchmark'] != 'MME'].set_index('Benchmark').drop(columns='Category')
best = gap_df.max(axis=1)

for model in MODELS:
    gap_df[f'{MODEL_SHORT[model]} gap'] = gap_df[model] - best

gap_cols = [f'{MODEL_SHORT[m]} gap' for m in MODELS]

fig, ax = plt.subplots(figsize=(12, 7))
gap_plot = gap_df[gap_cols].sort_values(gap_cols[0])
y = np.arange(len(gap_plot))
h = 0.25

for i, (col, model) in enumerate(zip(gap_cols, MODELS)):
    vals = gap_plot[col].values
    ax.barh(y + i * h, vals, h, label=MODEL_SHORT[model], color=COLORS[model], alpha=0.85)

ax.set_yticks(y + h)
ax.set_yticklabels(gap_plot.index)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_xlabel('Gap from Best Model (negative = lagging behind)')
ax.set_title('Performance Gap: How Far Each Model Lags Behind the Best')
ax.legend()
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Win/Loss summary across benchmarks
score_df = overall_df[overall_df['Benchmark'] != 'MME'].set_index('Benchmark')[MODELS]

wins = {}
for model in MODELS:
    wins[MODEL_SHORT[model]] = {
        'Wins (Best)': (score_df.idxmax(axis=1) == model).sum(),
        'Losses (Worst)': (score_df.idxmin(axis=1) == model).sum(),
        'Avg Score': score_df[model].mean(),
        'Avg Gap to Best': (score_df[model] - score_df.max(axis=1)).mean(),
    }

summary = pd.DataFrame(wins).T
summary['Avg Score'] = summary['Avg Score'].round(2)
summary['Avg Gap to Best'] = summary['Avg Gap to Best'].round(2)
display(summary)

# Where each model wins
for model in MODELS:
    benchmarks_won = score_df.index[score_df.idxmax(axis=1) == model].tolist()
    benchmarks_lost = score_df.index[score_df.idxmin(axis=1) == model].tolist()
    print(f"\n{MODEL_SHORT[model]}:")
    print(f"  Best at: {', '.join(benchmarks_won) if benchmarks_won else 'None'}")
    print(f"  Worst at: {', '.join(benchmarks_lost) if benchmarks_lost else 'None'}")

## 6. Per-Sample Error Analysis
For MCQ benchmarks, load per-sample predictions and analyze agreement/disagreement patterns.

In [ ]:
def load_predictions(benchmark):
    """Load per-sample predictions from .xlsx for all 3 models."""
    frames = {}
    for model in MODELS:
        fp = os.path.join(PATHS[model], FILE_PREFIX[model] + f'_{benchmark}.xlsx')
        if os.path.exists(fp):
            df = pd.read_excel(fp)
            frames[model] = df
    return frames


def analyze_mcq_agreement(benchmark, id_col='index', answer_col='answer', pred_col='prediction'):
    """Analyze agreement patterns for MCQ benchmarks."""
    preds = load_predictions(benchmark)
    if len(preds) < 2:
        print(f'Not enough models with predictions for {benchmark}')
        return None
    
    merged = None
    for model, df in preds.items():
        short = MODEL_SHORT[model]
        sub = df[[id_col, answer_col, pred_col]].copy()
        sub = sub.rename(columns={pred_col: f'pred_{short}'})
        if merged is None:
            merged = sub
        else:
            merged = merged.merge(sub[[id_col, f'pred_{short}']], on=id_col, how='inner')
    
    for model in MODELS:
        if model not in preds:
            continue
        short = MODEL_SHORT[model]
        pred_series = merged[f'pred_{short}'].astype(str).str.strip().str.upper().str[0]
        ans_series = merged[answer_col].astype(str).str.strip().str.upper().str[0]
        merged[f'correct_{short}'] = pred_series == ans_series
    
    correct_cols = [f'correct_{MODEL_SHORT[m]}' for m in MODELS if m in preds]
    merged['all_correct'] = merged[correct_cols].all(axis=1)
    merged['all_wrong'] = ~merged[correct_cols].any(axis=1)
    merged['n_correct'] = merged[correct_cols].sum(axis=1)
    
    total = len(merged)
    print(f'\n=== {benchmark} ({total} questions) ===')
    print(f"  All 3 correct:           {merged['all_correct'].sum():>5} ({merged['all_correct'].mean()*100:.1f}%)")
    print(f"  All 3 wrong:             {merged['all_wrong'].sum():>5} ({merged['all_wrong'].mean()*100:.1f}%)")
    print(f"  Exactly 1 correct:       {(merged['n_correct'] == 1).sum():>5} ({(merged['n_correct'] == 1).mean()*100:.1f}%)")
    print(f"  Exactly 2 correct:       {(merged['n_correct'] == 2).sum():>5} ({(merged['n_correct'] == 2).mean()*100:.1f}%)")
    
    for model in MODELS:
        if model not in preds:
            continue
        short = MODEL_SHORT[model]
        unique_correct = merged[f'correct_{short}'] & (merged['n_correct'] == 1)
        unique_wrong = ~merged[f'correct_{short}'] & (merged['n_correct'] == 2)
        print(f"  {short:>12} uniquely correct: {unique_correct.sum():>4} | uniquely wrong: {unique_wrong.sum():>4}")
    
    return merged


mcq_benchmarks = [
    ('AI2D_TEST', 'index', 'answer', 'prediction'),
    ('MMStar', 'index', 'answer', 'prediction'),
    ('MMMU_DEV_VAL', 'index', 'answer', 'prediction'),
    ('SEEDBench_IMG', 'index', 'answer', 'prediction'),
    ('RealWorldQA', 'index', 'answer', 'prediction'),
    ('ScienceQA_TEST', 'index', 'answer', 'prediction'),
]

agreement_results = {}
for bench, id_col, ans_col, pred_col in mcq_benchmarks:
    result = analyze_mcq_agreement(bench, id_col, ans_col, pred_col)
    if result is not None:
        agreement_results[bench] = result

In [ ]:
# Venn-style overlap visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for idx, (bench, merged) in enumerate(agreement_results.items()):
    if idx >= 6:
        break
    ax = axes[idx]
    
    categories = ['All correct', 'All wrong', 'Only 1 correct', 'Only 2 correct']
    values = [
        merged['all_correct'].mean() * 100,
        merged['all_wrong'].mean() * 100,
        (merged['n_correct'] == 1).mean() * 100,
        (merged['n_correct'] == 2).mean() * 100,
    ]
    colors_pie = ['#4CAF50', '#F44336', '#FF9800', '#8BC34A']
    
    wedges, texts, autotexts = ax.pie(values, labels=categories, autopct='%1.1f%%',
                                      colors=colors_pie, textprops={'fontsize': 7})
    for t in autotexts:
        t.set_fontsize(7)
    ax.set_title(bench, fontsize=10)

for idx in range(len(agreement_results), 6):
    axes[idx].set_visible(False)

plt.suptitle('Question Difficulty Distribution: How Many Models Get It Right?', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Per-model unique strengths and weaknesses from per-sample analysis
unique_data = []
for bench, merged in agreement_results.items():
    total = len(merged)
    for model in MODELS:
        short = MODEL_SHORT[model]
        if f'correct_{short}' not in merged.columns:
            continue
        acc = merged[f'correct_{short}'].mean() * 100
        unique_right = (merged[f'correct_{short}'] & (merged['n_correct'] == 1)).sum()
        unique_wrong = (~merged[f'correct_{short}'] & (merged['n_correct'] == 2)).sum()
        unique_data.append({
            'Benchmark': bench, 'Model': short,
            'Accuracy': acc,
            'Uniquely Correct': unique_right,
            'Uniquely Wrong': unique_wrong,
            'Unique Correct %': unique_right / total * 100,
            'Unique Wrong %': unique_wrong / total * 100,
        })

unique_df = pd.DataFrame(unique_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Uniquely correct
pivot_right = unique_df.pivot(index='Benchmark', columns='Model', values='Unique Correct %')
pivot_right.plot(kind='bar', ax=axes[0], color=[COLORS[m] for m in MODELS if MODEL_SHORT[m] in pivot_right.columns],
                 alpha=0.85, width=0.7)
axes[0].set_title('Uniquely Correct Questions (% of total)')
axes[0].set_ylabel('%')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=30, ha='right')
axes[0].grid(axis='y', alpha=0.3)

# Uniquely wrong
pivot_wrong = unique_df.pivot(index='Benchmark', columns='Model', values='Unique Wrong %')
pivot_wrong.plot(kind='bar', ax=axes[1], color=[COLORS[m] for m in MODELS if MODEL_SHORT[m] in pivot_wrong.columns],
                 alpha=0.85, width=0.7)
axes[1].set_title('Uniquely Wrong Questions (% of total)')
axes[1].set_ylabel('%')
axes[1].set_xticklabels(axes[1].get_xticklabels(), rotation=30, ha='right')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('Unique Strengths & Weaknesses per Model', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 7. MMMU Deep-Dive: Per-Subject Accuracy

In [ ]:
# MMMU per-subject (fine-grained) heatmap
df = load_subcategory_csv('MMMU_DEV_VAL', 'acc')
df = df[df['split'] == 'validation']

subject_cols = [c for c in df.columns if c not in (
    'split', 'model', 'Overall',
    'Art & Design', 'Business', 'Health & Medicine',
    'Humanities & Social Science', 'Science', 'Tech & Engineering'
)]

df_pivot = df.set_index('model')[subject_cols].astype(float) * 100
df_pivot.index = [MODEL_SHORT.get(m, m) for m in df_pivot.index]

fig, ax = plt.subplots(figsize=(22, 4))
im = ax.imshow(df_pivot.values, cmap='RdYlGn', aspect='auto')
ax.set_xticks(range(len(df_pivot.columns)))
ax.set_xticklabels([c.replace('_', '\n') for c in df_pivot.columns], rotation=60, ha='right', fontsize=7)
ax.set_yticks(range(len(df_pivot.index)))
ax.set_yticklabels(df_pivot.index)
for i in range(len(df_pivot.index)):
    for j in range(len(df_pivot.columns)):
        val = df_pivot.values[i, j]
        ax.text(j, i, f'{val:.0f}', ha='center', va='center', fontsize=6,
                color='white' if val < 35 else 'black')
plt.colorbar(im, ax=ax, shrink=0.6)
ax.set_title('MMMU (validation) — Per-Subject Accuracy')
plt.tight_layout()
plt.show()

# Find biggest gaps per subject
gaps = df_pivot.max(axis=0) - df_pivot.min(axis=0)
top_gaps = gaps.sort_values(ascending=False).head(10)
print('\nTop 10 subjects with largest model disagreement:')
for subj, gap in top_gaps.items():
    best_model = df_pivot[subj].idxmax()
    worst_model = df_pivot[subj].idxmin()
    print(f'  {subj:<35} gap={gap:.1f}pp  best={best_model} ({df_pivot[subj].max():.0f}) worst={worst_model} ({df_pivot[subj].min():.0f})')

## 8. MMBench Deep-Dive: Fine-Grained Capabilities

In [ ]:
_ = plot_subcategory_heatmap(
    'MMBench_DEV_EN', 'acc',
    exclude_cols={'AR', 'CP', 'FP-C', 'FP-S', 'LR', 'RR'},
    title='MMBench — Fine-Grained Skill Breakdown'
)

## 9. POPE Hallucination Analysis

In [ ]:
pope_data = {}
for model in MODELS:
    fp = os.path.join(PATHS[model], FILE_PREFIX[model] + '_POPE_score.csv')
    df = pd.read_csv(fp)
    pope_data[model] = df.set_index('split')['Overall']

pope_df = pd.DataFrame(pope_data)
pope_df.columns = [MODEL_SHORT[m] for m in MODELS]

fig, ax = plt.subplots(figsize=(10, 5))
pope_df.loc[['random', 'popular', 'adversarial']].plot(
    kind='bar', ax=ax, color=[COLORS[m] for m in MODELS], alpha=0.85, width=0.7)
ax.set_title('POPE — Hallucination Resistance by Difficulty')
ax.set_ylabel('Accuracy (%)')
ax.set_xticklabels(['Random', 'Popular', 'Adversarial'], rotation=0)
ax.set_ylim(75, 95)
ax.grid(axis='y', alpha=0.3)
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f', fontsize=8, padding=2)
plt.tight_layout()
plt.show()

print('\nPOPE drop from random → adversarial:')
for model in MODELS:
    short = MODEL_SHORT[model]
    drop = pope_df.loc['random', short] - pope_df.loc['adversarial', short]
    print(f'  {short}: {drop:.1f}pp')

## 10. Summary: Key Findings & Recommendations

In [ ]:
# Compute normalized rankings
rank_df = overall_df[overall_df['Benchmark'] != 'MME'].set_index('Benchmark')[MODELS]
rankings = rank_df.rank(axis=1, ascending=False)
avg_rank = rankings.mean()

print('=' * 70)
print('SUMMARY: Model Comparison Across 15 Benchmarks (excl. MME)')
print('=' * 70)

for model in MODELS:
    short = MODEL_SHORT[model]
    score_vals = rank_df[model].dropna()
    n_best = (rankings[model] == 1).sum()
    n_worst = (rankings[model] == 3).sum()
    
    print(f'\n{short} (avg rank: {avg_rank[model]:.2f}):')
    print(f'  Average score: {score_vals.mean():.2f}')
    print(f'  Best on {n_best} benchmarks, worst on {n_worst} benchmarks')
    
    # Find top 3 strengths (highest relative to others)
    rel_strength = rank_df[model] - rank_df.mean(axis=1)
    top_str = rel_strength.sort_values(ascending=False).head(3)
    print(f'  Relative strengths: {", ".join([f"{b} (+{v:.1f})" for b, v in top_str.items()])}')
    
    # Find top 3 weaknesses
    top_weak = rel_strength.sort_values().head(3)
    print(f'  Relative weaknesses: {", ".join([f"{b} ({v:.1f})" for b, v in top_weak.items()])}')

# Correlation between models
print('\n' + '=' * 70)
print('Model Score Correlation (how similar are the profiles):')
print('=' * 70)
corr = rank_df.corr()
corr.columns = [MODEL_SHORT[m] for m in MODELS]
corr.index = [MODEL_SHORT[m] for m in MODELS]
display(corr.round(3))

In [ ]:
# Final ranked comparison table (sorted by overall performance)
final_df = overall_df.copy()
final_df = final_df.set_index(['Benchmark', 'Category'])

def highlight_best(row):
    vals = row[MODELS]
    numeric_vals = pd.to_numeric(vals, errors='coerce')
    is_best = numeric_vals == numeric_vals.max()
    return ['font-weight: bold; color: green' if v else '' for v in is_best]

styled = final_df[MODELS].rename(columns={m: MODEL_SHORT[m] for m in MODELS})
styled = styled.style.apply(highlight_best, axis=1).format('{:.2f}', na_rep='N/A')
styled.set_caption('Complete Benchmark Results (green = best model)')
display(styled)